## Baseline: Fine-Tuning Naive

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
from utils import load_classifier
import torch.nn.functional as F
import torch
from train import train_classifier, evaluate_classifier,evaluate_task_incremental
from dataloaders import SequentialCIFAR10
from torch import nn
from models import LinearProbe, CNN

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")


In [5]:
seq_cifar = SequentialCIFAR10(data_root="./data", batch_size=32, buffer_size=200)

In [37]:
def train_and_test_task(task_nr, epochs = 2, criterion = nn.CrossEntropyLoss()):
    print(f"---------------- Training task {task_nr} -------------------------")
    train_dataloader, val_dataloader = seq_cifar.get_task_il_train_val_loaders(task_id=task_nr)
    model = load_classifier('checkpoints/task_0_pretrain', device=device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    train_classifier(
        model,
        train_dataloader,
        val_dataloader,
        optimizer,
        criterion,
        num_epochs=epochs,
        device=device,
    )
    test_loaders = seq_cifar.get_task_il_test_loaders(task_nr)
    
    task_results = evaluate_task_incremental(model, task_nr, test_loaders, criterion, device=device)
    for eval_task, results in task_results.items():
        print(f"Task {eval_task} - Test Loss: {results['loss']:.4f}, Test Accuracy: {results['acc']:.4f}")

    #save 
    torch.save(model.state_dict(), f'checkpoints/task_{task_nr}_model.pth')

In [38]:
for task in range(1, 5):
    train_and_test_task(task, epochs=2)

---------------- Training task 0 -------------------------
Backbone + linear head loaded successfully into reloaded_linear_probe.


Training:   0%|          | 0/2 [00:00<?, ?it/s]/Users/alexanderbodner/Documents/Udesa/5to/vision avanzada/tp1_continual_learning/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Training: 100%|██████████| 2/2 [00:54<00:00, 27.46s/it]


Task task_0 - Test Loss: 0.3275, Test Accuracy: 0.9070
---------------- Training task 1 -------------------------
Backbone + linear head loaded successfully into reloaded_linear_probe.


Training: 100%|██████████| 2/2 [00:55<00:00, 27.93s/it]


Task task_0 - Test Loss: 0.5097, Test Accuracy: 0.8625
Task task_1 - Test Loss: 0.6715, Test Accuracy: 0.5865
---------------- Training task 2 -------------------------
Backbone + linear head loaded successfully into reloaded_linear_probe.


...

Task task_0 - Test Loss: 0.5248, Test Accuracy: 0.8550
Task task_1 - Test Loss: 0.6755, Test Accuracy: 0.5755
Task task_2 - Test Loss: 0.6837, Test Accuracy: 0.5540
---------------- Training task 3 -------------------------
Backbone + linear head loaded successfully into reloaded_linear_probe.


Training: 100%|██████████| 2/2 [00:55<00:00, 27.87s/it]


Task task_0 - Test Loss: 0.9314, Test Accuracy: 0.1345
Task task_1 - Test Loss: 0.7387, Test Accuracy: 0.4525
Task task_2 - Test Loss: 0.7283, Test Accuracy: 0.4525
Task task_3 - Test Loss: 0.6600, Test Accuracy: 0.6140
---------------- Training task 4 -------------------------
Backbone + linear head loaded successfully into reloaded_linear_probe.


Training: 100%|██████████| 2/2 [00:55<00:00, 27.90s/it]


Task task_0 - Test Loss: 0.3681, Test Accuracy: 0.9010
Task task_1 - Test Loss: 0.6982, Test Accuracy: 0.5675
Task task_2 - Test Loss: 0.7067, Test Accuracy: 0.5500
Task task_3 - Test Loss: 0.8256, Test Accuracy: 0.4015
Task task_4 - Test Loss: 0.5280, Test Accuracy: 0.7540


In [ ]:
for task in range(1, 5):
    train_and_test_task(task, epochs=10)

# EWC

In [11]:
class EWCLoss(nn.Module):
    def __init__(self, lambda_reg=0.25):
        super().__init__()
        self.loss1 = nn.CrossEntropyLoss()
        self.lambda_ = lambda_reg

    def forward(self, x, labels, model_taskB, model_taskA):
        ewc_term_loss = 0
        ce_b = self.loss1(model_taskB(x), labels)

        logits_a = model_taskA(x)
        logp = F.log_softmax(logits_a, dim=1)[:, labels].mean()

        grads_a = torch.autograd.grad(
            logp, tuple(model_taskA.parameters()),
            retain_graph=False, create_graph=False, allow_unused=True
        )
        for param_b, param_a, grad_a in zip(
            model_taskB.parameters(), model_taskA.parameters(), grads_a
        ):
            fisher = grad_a ** 2
            ewc_term_loss += (fisher * (param_b - param_a) ** 2).sum()

        return ce_b + self.lambda_ * ewc_term_loss

In [12]:
from train_ewc import *

In [13]:
from copy import deepcopy
def train_and_test_task(task_nr, old_model,epochs = 2, criterion = nn.CrossEntropyLoss()):
    print(f"---------------- Training task {task_nr} -------------------------")
    train_dataloader, val_dataloader = seq_cifar.get_task_il_train_val_loaders(task_id=task_nr)
    new_model = deepcopy(old_model)

    # Enable gradients on all parameters
    for p in old_model.parameters():
        p.requires_grad_(True)
    optimizer = torch.optim.Adam(new_model.parameters(), lr=1e-3)
    train_classifier(
        new_model,
        train_dataloader,
        val_dataloader,
        optimizer,
        criterion,
        num_epochs=epochs,
        device=device,
        model_taskA=old_model,
    )
    test_loaders = seq_cifar.get_task_il_test_loaders(task_nr)

    task_results = evaluate_task_incremental(new_model, task_nr, test_loaders, criterion, device=device, model_taskA=old_model)
    for eval_task, results in task_results.items():
        print(f"Task {eval_task} - Test Loss: {results['loss']:.4f}, Test Accuracy: {results['acc']:.4f}")

    # save weights only (safe and compatible with torch>=2.6)
    import os
    os.makedirs('checkpoints/ewc', exist_ok=True)
    torch.save(new_model, f'checkpoints/ewc/task_{task_nr}_model.pth')

In [14]:
model_task0 = torch.load(f'checkpoints/ewc/task_{0}_model.pth', weights_only=False)#load_classifier('checkpoints/task_0_pretrain', device=device)

In [15]:
for task in range(1, 5):
    model_task0 = torch.load(f'checkpoints/ewc/task_{task-1}_model.pth' , weights_only=False)#load_classifier('checkpoints/task_0_pretrain', device=device)

    train_and_test_task(task, old_model=model_task0, epochs=5, criterion=EWCLoss(0.25))

---------------- Training task 1 -------------------------


Training:   0%|          | 0/5 [00:00<?, ?it/s]Python(35699) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(35704) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(35748) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(35756) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Training:  20%|██        | 1/5 [00:35<02:20, 35.14s/it]Python(35772) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(35773) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(35799) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(35800) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Training:  40%|████      | 2/5 [01:06<01:38, 32.88s/it]Python(35812) MallocStackLogging: can't turn off malloc sta

Task task_0 - Test Loss: 0.6922, Test Accuracy: 0.6625
Task task_1 - Test Loss: 0.4620, Test Accuracy: 0.7835
---------------- Training task 2 -------------------------


Training:   0%|          | 0/5 [00:00<?, ?it/s]Python(35865) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(35867) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(35870) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(35871) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Training:  20%|██        | 1/5 [00:31<02:04, 31.13s/it]Python(35876) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(35878) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(35885) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(35887) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Training:  40%|████      | 2/5 [01:02<01:33, 31.01s/it]Python(35890) MallocStackLogging: can't turn off malloc sta

Task task_0 - Test Loss: 0.7891, Test Accuracy: 0.6210
Task task_1 - Test Loss: 0.7784, Test Accuracy: 0.6675
Task task_2 - Test Loss: 0.3346, Test Accuracy: 0.8705
---------------- Training task 3 -------------------------


Training:   0%|          | 0/5 [00:00<?, ?it/s]Python(35959) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(35960) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(35968) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(35969) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Training:  20%|██        | 1/5 [00:30<02:02, 30.67s/it]Python(35972) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(35973) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(35980) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(35981) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Training:  40%|████      | 2/5 [01:01<01:32, 30.71s/it]Python(35985) MallocStackLogging: can't turn off malloc sta

Task task_0 - Test Loss: 1.2950, Test Accuracy: 0.5340
Task task_1 - Test Loss: 1.3192, Test Accuracy: 0.5345
Task task_2 - Test Loss: 1.0877, Test Accuracy: 0.6150
Task task_3 - Test Loss: 0.1741, Test Accuracy: 0.9425
---------------- Training task 4 -------------------------


Training:   0%|          | 0/5 [00:00<?, ?it/s]Python(36056) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(36060) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(36066) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(36067) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Training:  20%|██        | 1/5 [00:30<02:02, 30.74s/it]Python(36070) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(36071) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(36074) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(36075) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Training:  40%|████      | 2/5 [01:01<01:31, 30.55s/it]Python(36078) MallocStackLogging: can't turn off malloc sta

Task task_0 - Test Loss: 0.6091, Test Accuracy: 0.7545
Task task_1 - Test Loss: 1.2623, Test Accuracy: 0.5080
Task task_2 - Test Loss: 1.2577, Test Accuracy: 0.5135
Task task_3 - Test Loss: 1.0972, Test Accuracy: 0.6020
Task task_4 - Test Loss: 0.2647, Test Accuracy: 0.8960


In [ ]:
len(list(model_task0.parameters())[-1])

2